# 1. 결정 트리 훈련

1-a . make_moons (n_sample=1000, noise=0.4)를 사용해 데이터셋을 생성합니다.

In [1]:
from sklearn.datasets import make_moons

X,y = make_moons(n_samples=1000, noise=0.4, random_state=42)

1-b. 이를 train_test_split( )을 사용해 훈련 세트와 테스트 세트로 나눕니다.(random_state=42, train:test = 8:2 비율로)


In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

1-c. DecisionTreeClassifier의 최적의 매개변수를 찾기 위해 교차 검증과 함께 그리드 탐색을 수행합니다. (GridSearchCV)

힌트 : max_depth, min_samples_split, max_leaf_nodes 등 하이퍼파라미터를 다양하게 적용해보세요! (교재 pg. 113, 207 참고)

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

# 예시 파라미터: params = {'max_depth': [3,5,7,9], 'min_samples_split' : [2,3,4], 'max_leaf_nodes' : list(range(2,100))}

In [4]:
params = {'max_depth' : [3 , 5 , 7 , 9 ] , 'min_samples_split' : [2 , 3 , 4 ] , 'max_leaf_nodes' : list (range (2 , 100 ) ) } 

tree_clf = DecisionTreeClassifier (random_state = 42 ) 
grid_search_cv = GridSearchCV (tree_clf , params , cv = 5 , scoring = 'accuracy' , return_train_score = True ) 
grid_search_cv . fit (X_train , y_train ) 

# 최적의 매개변수 확인 (print문으로 감싸거나 세미콜론을 붙여 HTML 렌더링 우회)
print(grid_search_cv.best_estimator_)



DecisionTreeClassifier(max_depth=5, max_leaf_nodes=6, random_state=42)


1-d. 찾은 매개변수를 사용해 전체 훈련 세트에 대해 모델을 훈련시키고 테스트 세트에서 성능(정확도)을 측정합니다. (대략 85~87% 정도 나옵니다.)

In [5]:
# GridSearchCV는 최적의 모델로 다시 훈련시키기 때문에 별도의 훈련이 필요없습니다.

from sklearn.metrics import accuracy_score

# GridSearchCV 객체는 이미 최적의 매개변수로 전체 훈련 세트(X_train, y_train)에 대한 학습이 완료되어 있습니다.
# 바로 테스트 세트(X_test)에 대한 예측을 수행하고 정확도를 측정합니다.
y_pred = grid_search_cv.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))

Test Accuracy: 0.855


# 2. 랜덤포레스트

In [6]:
import urllib.request
import zipfile
import os
import ssl

# SSL 인증서 검증 우회 설정
ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

url = 'https://bit.ly/3i4n1QB'
req = urllib.request.Request(url)

with urllib.request.urlopen(req, context=ctx) as response, open('3i4n1QB', 'wb') as out_file:
    out_file.write(response.read())

os.makedirs('data', exist_ok=True)
with zipfile.ZipFile('3i4n1QB', 'r') as existing_zip:
    existing_zip.extractall('data')

In [7]:
# 라이브러리 불러오기

import pandas as pd
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

In [8]:
train = pd.read_csv('data/train.csv')

# Scaling
scaler = MinMaxScaler()
scaler.fit(train[['fixed acidity']])
train['Scaled fixed acidity'] = scaler.transform(train[['fixed acidity']])

# Encoding
encoder = OneHotEncoder()
encoder.fit(train[['type']])
onehot = encoder.transform(train[['type']])
onehot = onehot.toarray()
onehot = pd.DataFrame(onehot)
onehot.columns = encoder.get_feature_names_out()
train = pd.concat([train, onehot], axis = 1)
train = train.drop(columns = ['type'])
train.head()

,index,quality,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,Scaled fixed acidity,type_red,type_white
0,0,5,5.6,0.695,0.06,6.8,0.042,9.0,84.0,0.99432,3.44,0.44,10.2,0.148760,0.0,1.0
1,1,5,8.8,0.610,0.14,2.4,0.067,10.0,42.0,0.99690,3.19,0.59,9.5,0.413223,1.0,0.0
2,2,5,7.9,0.210,0.39,2.0,0.057,21.0,138.0,0.99176,3.05,0.52,10.9,0.338843,0.0,1.0
3,3,6,7.0,0.210,0.31,6.0,0.046,29.0,108.0,0.99390,3.26,0.50,10.8,0.264463,0.0,1.0
4,4,6,7.8,0.400,0.26,9.5,0.059,32.0,178.0,0.99550,3.04,0.43,10.9,0.330579,0.0,1.0


2-a. 랜덤포레스트 분류 모형을 불러옵니다.

In [9]:
from sklearn.ensemble import RandomForestClassifier

2-b. 랜덤포레스트 분류 모형을 "random_classifier"라는 변수에 저장합니다.

In [10]:
random_classifier = RandomForestClassifier(random_state=42)

2-c. "X"라는 변수에 train의 "quality" 피쳐를 제거하고 저장합니다.

In [11]:
X = train.drop(columns=['quality'])

2-d. "y"라는 변수에 정답인 train의 "quality" 피쳐를 저장합니다.

In [12]:
y = train['quality']

2-e. "X"와 "y"를 train set과 test set으로 분할합니다. (test_size=0.2)

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

2-f. "random_classifier"를 X_train과 y_train을 이용해 학습시켜보세요. 이후, 학습시킨 모델로 X_test 값을 통해 y_pred 값을 예측해보세요.

In [14]:
random_classifier.fit(X_train, y_train)
y_pred = random_classifier.predict(X_test)

2-g. 예측된 결과를 confusion matrix와 Accuracy를 이용해 평가해보세요.

In [15]:
from sklearn.metrics import confusion_matrix, accuracy_score

print(confusion_matrix(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

[[  0   0   1   2   0   0   0]
 [  0   3  18  14   0   0   0]
 [  0   1 246  99   5   0   0]
 [  0   0  77 387  29   0   0]
 [  0   0   3  84  95   1   0]
 [  0   0   0  14   8  12   0]
 [  0   0   0   1   0   0   0]]
Accuracy: 0.6754545454545454
